## Setup & ICD-10 Reference Table

In [5]:
import re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white", "axes.spines.top":False,"axes.spines.right":False})
ICD10_REF = pd.DataFrame([
    {"code":"I50.20","desc":"Unspecified systolic heart failure",
     "kw":"systolic heart failure reduced ejection fraction HFrEF EF low BNP furosemide"},
    {"code":"I50.30","desc":"Unspecified diastolic heart failure",
     "kw":"diastolic heart failure preserved ejection HFpEF"},
    {"code":"I50.9", "desc":"Unspecified heart failure",
     "kw":"heart failure congestive CHF decompensated BNP edema"},
    {"code":"I21.19","desc":"STEMI other coronary artery",
     "kw":"STEMI myocardial infarction ST elevation troponin cath lab PCI stent"},
    {"code":"I21.4", "desc":"Non-ST elevation MI",
     "kw":"NSTEMI troponin elevated demand ischemia non-ST elevation ACS"},
    {"code":"I48.91","desc":"Unspecified atrial fibrillation",
     "kw":"atrial fibrillation AFib arrhythmia rate control anticoagulation"},
    {"code":"J44.1", "desc":"COPD with acute exacerbation",
     "kw":"COPD exacerbation bronchospasm wheezes steroids albuterol ipratropium"},
    {"code":"J18.9", "desc":"Pneumonia unspecified",
     "kw":"pneumonia consolidation fever WBC antibiotics community acquired CAP"},
    {"code":"J26.99","desc":"Other pulmonary embolism",
     "kw":"pulmonary embolism PE DVT anticoagulation rivaroxaban heparin RV strain"},
    {"code":"E11.10","desc":"T2DM with DKA without coma",
     "kw":"diabetic ketoacidosis DKA anion gap metabolic acidosis insulin drip glucose"},
    {"code":"E11.65","desc":"T2DM with hyperglycemia",
     "kw":"uncontrolled diabetes hyperglycemia HbA1c elevated glucose A1c"},
    {"code":"N17.9", "desc":"Acute kidney failure",
     "kw":"acute kidney injury AKI creatinine elevated BUN nephrotoxins oliguria"},
    {"code":"N18.6", "desc":"End-stage renal disease",
     "kw":"ESRD hemodialysis dialysis end stage renal disease potassium"},
    {"code":"A41.9", "desc":"Sepsis unspecified organism",
     "kw":"sepsis septic shock bacteremia lactate broad spectrum antibiotics norepinephrine"},
    {"code":"I10",   "desc":"Essential hypertension",
     "kw":"hypertension HTN blood pressure elevated antihypertensive"},
])

print(f"ICD-10 reference: {len(ICD10_REF)} codes")
display(ICD10_REF[["code","desc"]].head(8))

ICD-10 reference: 15 codes


,code,desc
0,I50.20,Unspecified systolic heart failure
1,I50.30,Unspecified diastolic heart failure
2,I50.9,Unspecified heart failure
3,I21.19,STEMI other coronary artery
4,I21.4,Non-ST elevation MI
5,I48.91,Unspecified atrial fibrillation
6,J44.1,COPD with acute exacerbation
7,J18.9,Pneumonia unspecified


## TF-IDF ICD-10 Code Suggester

In [10]:
tfidf_icd = TfidfVectorizer(ngram_range=(1,2), max_features=500)
icd_matrix = tfidf_icd.fit_transform(ICD10_REF["kw"]+" "+ ICD10_REF["desc"])

def suggest_icd(text, top_k=5, threshold=0.12):
    vec = tfidf_icd.transform([text.lower()])
    scores = cosine_similarity(vec, icd_matrix).flatten()
    top_idx = scores.argsort()[-top_k:][::-1]
    return [{"code":ICD10_REF.iloc[i]["code"], "desc":ICD10_REF.iloc[i]["desc"], "score":round(float(scores[i]),3)} for i in top_idx if scores[i]>=threshold]

test_notes = [
    "Acute decompensated heart failure, BNP 1450, EF 30%, IV furosemide started.",
    "STEMI inferior wall, troponin 2.4, EKG ST elevation, PCI with stent placement.",
    "COPD exacerbation, diffuse wheezes, albuterol nebs, methylprednisolone IV.",
    "Diabetic ketoacidosis, glucose 520, anion gap 24, insulin drip initiated.",
    "Septic shock, lactate 4.2, norepinephrine, vancomycin and meropenem started.",
]

print("ICD-10 code Suggestions:\n")
for note in test_notes:
    suggestions = suggest_icd(note, top_k=3)
    print(f" Note: '{note[:60]}'")
    for s in suggestions:
        print(f" {s['code']:8s} ({s['score']:.3f}) - {s['desc']}")
    print()

ICD-10 code Suggestions:

 Note: 'Acute decompensated heart failure, BNP 1450, EF 30%, IV furo'
 I50.9    (0.475) - Unspecified heart failure
 I50.20   (0.432) - Unspecified systolic heart failure
 I50.30   (0.284) - Unspecified diastolic heart failure

 Note: 'STEMI inferior wall, troponin 2.4, EKG ST elevation, PCI wit'
 I21.19   (0.518) - STEMI other coronary artery
 I21.4    (0.367) - Non-ST elevation MI

 Note: 'COPD exacerbation, diffuse wheezes, albuterol nebs, methylpr'
 J44.1    (0.634) - COPD with acute exacerbation

 Note: 'Diabetic ketoacidosis, glucose 520, anion gap 24, insulin dr'
 E11.10   (0.572) - T2DM with DKA without coma

 Note: 'Septic shock, lactate 4.2, norepinephrine, vancomycin and me'
 A41.9    (0.455) - Sepsis unspecified organism



## Structured Medication Extraction

In [14]:
DRUG_LIST = sorted([
    "metoprolol","carvedilol","bisoprolol","atenolol","labetalol",
    "lisinopril","enalapril","ramipril","losartan","valsartan","sacubitril",
    "amlodipine","nifedipine","diltiazem","verapamil",
    "furosemide","torsemide","spironolactone","hydrochlorothiazide",
    "warfarin","rivaroxaban","apixaban","dabigatran","edoxaban",
    "aspirin","clopidogrel","ticagrelor","prasugrel",
    "heparin","enoxaparin","fondaparinux",
    "digoxin","amiodarone","sotalol",
    "atorvastatin","rosuvastatin","simvastatin","pravastatin",
    "nitroglycerin","nitroprusside","norepinephrine","vasopressin",
    "vancomycin","linezolid","ceftriaxone","cefazolin","cefepime",
    "piperacillin-tazobactam","meropenem","imipenem","ertapenem",
    "azithromycin","clarithromycin","doxycycline","ciprofloxacin","levofloxacin",
    "trimethoprim-sulfamethoxazole","nitrofurantoin",
    "metformin","glipizide","glimepiride","empagliflozin","dapagliflozin",
    "sitagliptin","liraglutide","semaglutide",
    "insulin","insulin glargine","insulin detemir","insulin aspart","insulin lispro",
    "albuterol","ipratropium","tiotropium","fluticasone","salmeterol","budesonide",
    "prednisone","methylprednisolone","dexamethasone","hydrocortisone",
    "omeprazole","pantoprazole","lansoprazole","famotidine",
    "ondansetron","metoclopramide","acetaminophen","ibuprofen","ketorolac",
    "morphine","hydromorphone","fentanyl","oxycodone","tramadol",
    "midazolam","lorazepam","propofol","dexmedetomidine",
    "fluconazole","micafungin","acyclovir","levothyroxine",
], key=len, reverse=True)


DOSE_UNIT = r'(?:mg|mcg|g|mL|units?|IU|mEq)'
ROUTE_PAT = r'(?:PO|IV|IM|SQ|SC|SL|inhaled|topical|PR)'
FREQ_PAT = r'(?:QD|BID|TID|QID|QAM|QHS|q\d+h|PRN|once daily|twice daily|daily|weekly)'
DRUG_PAT = r'\b(' + '|'. join(re.escape(d) for d in DRUG_LIST) + r')\b'
MED_PAT = (DRUG_PAT 
           + r'(?:[\s]+(\d+(?:\.\d+)?)\s*('+ DOSE_UNIT + r'))?'
           + r'(?:[\s,]+(' + ROUTE_PAT + r'))?'
           +r'(?:[\s,]+(' + FREQ_PAT + r'))?')

def extract_meds(text):
    seen = set(); meds = []
    for m in re.finditer(MED_PAT, text, re.IGNORECASE):
        drug = m.group(1).lower()
        if drug not in seen:
            seen.add(drug)
            meds.append({"drug":drug,"dose":(m.group(2) or "").strip(),
            "unit":(m.group(3) or "").strip(),
            "route":(m.group(4) or "").upper(),
            "freq":(m.group(5) or "").upper()})
    return meds

test_med_notes = [
    "Patient is on metoprolol 50 mg PO BID, lisinopril 10 mg PO QD, furosemide 40 mg PO QAM, warfarin 5 mg PO QD.",
    "Started vancomycin 1250 mg IV q12h, piperacillin-tazobactam 4.5 g IV q8h, fluconazole 400 mg IV QD.",
    "Insulin glargine 20 units SQ QHS, metformin 1000 mg PO BID, empagliflozin 10 mg PO QD.",
]
print("Structured medication extraction:\n")

for note in test_med_notes:
    meds = extract_meds(note)
    print(f"Note: '{note[:65]}' ")
    for m in meds:
        print(f" {m['drug']:30s} | {m['dose']:6s} | {m['route']:4s} | {m['freq']}")
    print()

Structured medication extraction:

Note: 'Patient is on metoprolol 50 mg PO BID, lisinopril 10 mg PO QD, fu' 
 metoprolol                     | 50     | PO   | BID
 lisinopril                     | 10     | PO   | QD
 furosemide                     | 40     | PO   | QAM
 warfarin                       | 5      | PO   | QD

Note: 'Started vancomycin 1250 mg IV q12h, piperacillin-tazobactam 4.5 g' 
 vancomycin                     | 1250   | IV   | Q12H
 piperacillin-tazobactam        | 4.5    | IV   | Q8H
 fluconazole                    | 400    | IV   | QD

Note: 'Insulin glargine 20 units SQ QHS, metformin 1000 mg PO BID, empag' 
 insulin glargine               | 20     | SQ   | QHS
 metformin                      | 1000   | PO   | BID
 empagliflozin                  | 10     | PO   | QD



## Medication Reconciliation

In [18]:
def reconcile_medications(admit_text, dc_text):
    admit_meds = {m["drug"]:m for m in extract_meds(admit_text)}
    dc_meds    = {m["drug"]:m for m in extract_meds(dc_text)}
    admit_set = set(admit_meds.keys())
    dc_set    = set(dc_meds.keys())
    continued = sorted(admit_set & dc_set)
    discontinued = sorted(admit_set - dc_set)
    new_at_dc = sorted(dc_set - admit_set)
    dose_changed = sorted(d for d in continued if admit_meds[d]["dose"] != dc_meds[d]["dose"]
                          and admit_meds[d]["dose"] and dc_meds[d]["dose"])
    return {"continued":continued,"discontinued":discontinued,"new_at_dc":new_at_dc,"dose_changed":dose_changed,"admit_meds":admit_meds,"dc_meds":dc_meds}
ADMIT_MEDS = (
    "metoprolol 50 mg PO BID, lisinopril 10 mg PO QD, furosemide 40 mg PO QAM, "
    "warfarin 5 mg PO QD, atorvastatin 40 mg PO QHS, metformin 1000 mg PO BID, aspirin 81 mg PO QD."
)
DC_MEDS = (
    "metoprolol 25 mg PO BID, lisinopril 10 mg PO QD, furosemide 80 mg PO QAM, "
    "rivaroxaban 20 mg PO QD, atorvastatin 40 mg PO QHS, metformin 1000 mg PO BID, "
    "aspirin 81 mg PO QD, spironolactone 25 mg PO QD, sacubitril 24 mg PO BID."
)

recon = reconcile_medications(ADMIT_MEDS, DC_MEDS)
print("MEDICATION RECONCILIATION REPORT")
print("="*55)
print(f"Admission: {len(recon['admit_meds'])} meds | Discharge: {len(recon['dc_meds'])} meds")
print(f"\nContinued ({len(recon['continued'])}):")
for m in recon["continued"]: print(f"  checkmark {m}")
print(f"\nDiscontinued ({len(recon['discontinued'])}):")
for m in recon["discontinued"]: print(f"  removed   {m}")
print(f"\nNew at discharge ({len(recon['new_at_dc'])}):")
for m in recon["new_at_dc"]: print(f"  added     {m}")
print(f"\nDose changed ({len(recon['dose_changed'])}):")
for m in recon["dose_changed"]:
    a = recon["admit_meds"][m]; d = recon["dc_meds"][m]
    print(f"  changed   {m}: {a['dose']} {a['unit']} -> {d['dose']} {d['unit']}")

MEDICATION RECONCILIATION REPORT
Admission: 7 meds | Discharge: 9 meds

Continued (6):
  checkmark aspirin
  checkmark atorvastatin
  checkmark furosemide
  checkmark lisinopril
  checkmark metformin
  checkmark metoprolol

Discontinued (1):
  removed   warfarin

New at discharge (3):
  added     rivaroxaban
  added     sacubitril
  added     spironolactone

Dose changed (2):
  changed   furosemide: 40 mg -> 80 mg
  changed   metoprolol: 50 mg -> 25 mg


## Drug-Drug Interaction Checker

In [20]:
DDI_REF = [
    ("warfarin",    "aspirin",          "HIGH", "Increased bleeding risk — monitor INR closely"),
    ("warfarin",    "ciprofloxacin",    "HIGH", "CYP2C9 inhibition — warfarin effect increased"),
    ("warfarin",    "fluconazole",      "HIGH", "Azole antifungals inhibit warfarin metabolism"),
    ("digoxin",     "amiodarone",       "HIGH", "Digoxin toxicity — reduce dose by 50%"),
    ("heparin",     "enoxaparin",       "HIGH", "Concurrent anticoagulants — serious bleeding risk"),
    ("metformin",   "contrast",         "MOD",  "Hold 48h post-contrast — nephropathy risk"),
    ("lisinopril",  "spironolactone",   "MOD",  "Hyperkalaemia risk — monitor K+ closely"),
    ("metoprolol",  "diltiazem",        "MOD",  "Additive bradycardia and hypotension"),
    ("aspirin",     "ibuprofen",        "MOD",  "NSAIDs reduce aspirin cardioprotection"),
]

def check_ddi(medication_list):
    med_names = {m["drug"].lower() for m in medication_list}
    flagged = []
    for d1,d2,severity,msg in DDI_REF:
        if d1 in med_names and d2 in med_names:
            flagged.append({"drug1":d1,"drug2":d2,"severity":severity,"message":msg})
    return flagged

complex_note = (
    "warfarin 5 mg QD, aspirin 81 mg QD, metformin 500 mg BID, "
    "lisinopril 10 mg QD, spironolactone 25 mg QD, digoxin 0.125 mg QD, "
    "amiodarone 200 mg QD, fluconazole 400 mg IV QD."
)

meds_list = extract_meds(complex_note)
interactions = check_ddi(meds_list)
print(f"Medications: {[m['drug'] for m in meds_list]}")
print(f"\nDrug-Drug Interactions ({len(interactions)} flagged):\n")
for i in interactions:
    icon = "HIGH_RISK" if i["severity"]=="HIGH" else "MOD_RISK"
    print(f"  [{icon}] {i['drug1']} + {i['drug2']}")
    print(f"           {i['message']}")

Medications: ['warfarin', 'aspirin', 'metformin', 'lisinopril', 'spironolactone', 'digoxin', 'amiodarone', 'fluconazole']

Drug-Drug Interactions (4 flagged):

  [HIGH_RISK] warfarin + aspirin
           Increased bleeding risk — monitor INR closely
  [HIGH_RISK] warfarin + fluconazole
           Azole antifungals inhibit warfarin metabolism
  [HIGH_RISK] digoxin + amiodarone
           Digoxin toxicity — reduce dose by 50%
  [MOD_RISK] lisinopril + spironolactone
           Hyperkalaemia risk — monitor K+ closely


## ICD-10 Code Validation

In [25]:
def validate_icd(code):
    code = str(code).strip().upper()
    valid = bool(
        re.match(r'^[A-Z]\d{2}(\.\d{1,4})?$', code)
    )
    match = ICD10_REF[ICD10_REF["code"]==code]
    return {"code":code,"valid_format":valid,"in_reference":len(match)>0, "desc":match.iloc[0]["desc"] if len(match)>0 else  "Not in Reference"}
test_codes = ["I50.9","E11.10","A41.9","INVALID","I10","XYZ1","J44.1","N17.9","I50.20"]
print("ICD-10 Code Validation:\n")
print(f"{'Code':10s} {'Format':10s} {'In Ref':8s} {'Description'}")
print("-"*65)
for code in test_codes:
    r = validate_icd(code)
    fmt = "Valid" if r["valid_format"] else "INVALID"
    ref = "Yes"  if r["in_reference"] else "No"
    print(f"  {code:8s} {fmt:10s} {ref:8s} {r['desc'][:45]}")    

ICD-10 Code Validation:

Code       Format     In Ref   Description
-----------------------------------------------------------------
  I50.9    Valid      Yes      Unspecified heart failure
  E11.10   Valid      Yes      T2DM with DKA without coma
  A41.9    Valid      Yes      Sepsis unspecified organism
  INVALID  INVALID    No       Not in Reference
  I10      Valid      Yes      Essential hypertension
  XYZ1     INVALID    No       Not in Reference
  J44.1    Valid      Yes      COPD with acute exacerbation
  N17.9    Valid      Yes      Acute kidney failure
  I50.20   Valid      Yes      Unspecified systolic heart failure
